# 1. Dataset description

The **Facebook Live Sellers in Thailand** dataset contains posts from the Facebook pages of 10 Thai fashion and cosmetics sellers, including post types and engagement metrics.

## 1.1. Environment and Library Imports

In [34]:
%pip install ucimlrepo -q

Note: you may need to restart the kernel to use updated packages.


In [35]:
# Standard library imports
from pathlib import Path
import sys

current_directory = Path.cwd().resolve()
project_candidates = (current_directory, *current_directory.parents)
PROJECT_ROOT = next(
    (path for path in project_candidates if (path / "src").is_dir()),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate the project root directory.")

SRC_DIRECTORY = PROJECT_ROOT / "src"

if str(SRC_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SRC_DIRECTORY))

### 1.1.1. Custom Utilities

In [36]:
from data_collection_utils import (
    dataframe_overview,
    fetch_uci_features,
    save_dataframe_csv,
    validate_dataframe_contract,
)

## 1.2. Data Loading

The UCI Machine Learning Repository is the primary data source. Because this is an unsupervised learning project, the collected feature table is named `raw_posts` rather than `X`, and no supervised target variable is defined.

In [37]:
UCI_DATASET_ID = 488

raw_posts = fetch_uci_features(UCI_DATASET_ID)

print("Dataset downloaded successfully.")
print(f"Shape: {raw_posts.shape}")

Dataset downloaded successfully.
Shape: (7050, 11)


### 1.2.1. Data Contract Validation

Validate the expected columns immediately after collection so that changes in the external data source fail early with a clear message.

In [38]:
REQUIRED_COLUMNS = {
    "status_type",
    "status_published",
    "num_reactions",
    "num_comments",
    "num_shares",
    "num_likes",
    "num_loves",
    "num_wows",
    "num_hahas",
    "num_sads",
    "num_angrys",
}

validate_dataframe_contract(
    raw_posts,
    required_columns=REQUIRED_COLUMNS,
)

print("Data contract validated successfully.")

Data contract validated successfully.


### 1.2.2. Saving the Raw Dataset

In [39]:
raw_data_path = save_dataframe_csv(
    raw_posts,
    PROJECT_ROOT / "data" / "raw" / "facebook_live_sellers.csv",
)

print(f"Raw dataset saved to: {raw_data_path}")

Raw dataset saved to: C:\Projetos\facebook_clusterizacao\data\raw\facebook_live_sellers.csv


### 1.2.3. Initial Inspection

This stage comprises the initial contact with the raw dataset (*Facebook Live Sellers Dataset*). The objective is to validate the dimensions, verify the mapped data types, and check the initial consistency of the structured variables.

#### 1.2.3.1. Dataset Metadata
| Characteristic | Detail |
| :--- | :--- |
| **Data Type** | Multivariate |
| **Subject Area** | Business / E-commerce |
| **ML Task** | Clustering |
| **Number of Instances** | 7,050 |
| **Predictive Attributes** | 11 |
| **Missing Values** | None (100% complete dataset) |

In [40]:
# Structural verification and data types
raw_posts.info()

display(dataframe_overview(raw_posts))

<class 'pandas.DataFrame'>
RangeIndex: 7050 entries, 0 to 7049
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   status_type       7050 non-null   str  
 1   status_published  7050 non-null   str  
 2   num_reactions     7050 non-null   int64
 3   num_comments      7050 non-null   int64
 4   num_shares        7050 non-null   int64
 5   num_likes         7050 non-null   int64
 6   num_loves         7050 non-null   int64
 7   num_wows          7050 non-null   int64
 8   num_hahas         7050 non-null   int64
 9   num_sads          7050 non-null   int64
 10  num_angrys        7050 non-null   int64
dtypes: int64(9), str(2)
memory usage: 606.0 KB


,dtype,non_null,missing,missing_pct,unique
column,,,,,
status_type,str,7050,0,0.0,4
status_published,str,7050,0,0.0,6913
num_reactions,int64,7050,0,0.0,1067
num_comments,int64,7050,0,0.0,993
num_shares,int64,7050,0,0.0,501
num_likes,int64,7050,0,0.0,1044
num_loves,int64,7050,0,0.0,229
num_wows,int64,7050,0,0.0,65
num_hahas,int64,7050,0,0.0,42


#### 1.2.3.2. Sample Visualization (Top 5 Records)
Below are the first five rows of the dataframe to understand the scale and original format of the data before any transformation is applied.

In [41]:
raw_posts.head()

,status_type,status_published,num_reactions,num_comments,num_shares,num_likes,num_loves,num_wows,num_hahas,num_sads,num_angrys
0,video,4/22/2018 6:00,529,512,262,432,92,3,1,1,0
1,photo,4/21/2018 22:45,150,0,0,150,0,0,0,0,0
2,video,4/21/2018 6:17,227,236,57,204,21,1,1,0,0
3,photo,4/21/2018 2:29,111,0,0,111,0,0,0,0,0
4,photo,4/18/2018 3:22,213,0,0,204,9,0,0,0,0


#### 1.2.3.3. Data Dictionary (Attributes)

Below is the detailed technical specification for each of the 11 columns present in the dataset:

| # | Attribute | Data Type | Description |
| :-: | :--- | :---: | :--- |
| **0** | `status_type` | `str` (Nominal) | Type of publication on the platform (*video, photo, status, link*). |
| **1** | `status_published` | `str` (Temporal) | Exact publication date and time of the post. |
| **2** | `num_reactions` | `int64` | Total count of consolidated reactions on the post. |
| **3** | `num_comments` | `int64` | Total number of comments made on the publication. |
| **4** | `num_shares` | `int64` | Number of times the post was shared. |
| **5** | `num_likes` | `int64` | Specific count of "Like" reactions. |
| **6** | `num_loves` | `int64` | Specific count of "Love" reactions. |
| **7** | `num_wows` | `int64` | Specific count of "Wow" reactions. |
| **8** | `num_hahas` | `int64` | Specific count of "Haha" reactions. |
| **9** | `num_sads` | `int64` | Specific count of "Sad" reactions. |
| **10** | `num_angrys` | `int64` | Specific count of "Angry" reactions. |